# Phase 4 — Real-model experiment pipeline

Pipeline theo từng cell: load repository mới nhất, load model thật, sanity check, benchmark từng variant, vẽ chart và xuất report. Chạy lần lượt để quan sát output sau mỗi bước.

In [ ]:
# Cell 1 — environment and repository
!nvidia-smi
!pip install -q transformers accelerate matplotlib seaborn pandas
import os, sys, subprocess, gc, json, time, random, datetime
from pathlib import Path
REPO_URL = os.environ.get('PROJECTIII_REPO_URL', 'https://github.com/Dung34/vector-kv-cache.git')
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT / 'vector-kv-cache'
    if not (ROOT / 'src').exists(): subprocess.run(['git','clone','--depth','1',REPO_URL,str(ROOT)], check=True)
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.') : del sys.modules[name]
sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)


In [ ]:
# Cell 2 — imports, seed and device
import torch, transformers, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.baselines.full_cache import FullKVCacheWrapper
from src.baselines.h2o_cache import H2OKVCache
from src.baselines.h2o_hook import H2OAttentionHookManager
from src.vector_cache.vector_cache import VectorKVCache
from src.vector_cache.vector_hook import VectorAttentionHookManager
SEED=42; random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'GPU is required for this notebook'
print({'device':torch.cuda.get_device_name(0),'torch':torch.__version__,'transformers':transformers.__version__,'seed':SEED})

In [ ]:
# Cell 3 — load a real model
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, attn_implementation='eager', trust_remote_code=True).to(DEVICE).eval()
print({'model':MODEL_ID,'dtype':str(DTYPE),'layers':model.config.num_hidden_layers,'hidden_size':model.config.hidden_size,'kv_heads':getattr(model.config,'num_key_value_heads',None)})

In [ ]:
# Cell 4 — prompt and model sanity check
PROMPT = 'Explain in three concise sentences why bounded KV cache memory matters for long-context inference.'
inputs = tokenizer(PROMPT, return_tensors='pt').to(DEVICE)
with torch.inference_mode():
    sanity = model.generate(**inputs, max_new_tokens=20, do_sample=False, use_cache=True)
print(tokenizer.decode(sanity[0], skip_special_tokens=True))
print({'prompt_tokens':inputs.input_ids.shape[-1],'output_tokens':sanity.shape[-1]})

In [ ]:
# Cell 5 — benchmark one variant
def make_cache(name, compression_ratio=0.5, target_budget=None, max_physical_budget=None):
    if name == 'full_fp16': return FullKVCacheWrapper(), None
    if name == 'h2o': return H2OKVCache(compression_ratio=compression_ratio,target_budget=target_budget,max_physical_budget=max_physical_budget,min_recent_tokens=16), H2OAttentionHookManager
    head_dim=model.config.hidden_size // model.config.num_attention_heads
    kv_heads=getattr(model.config,'num_key_value_heads',model.config.num_attention_heads)
    w_ols=torch.eye(head_dim,device=DEVICE,dtype=DTYPE).view(1,1,head_dim,head_dim).expand(model.config.num_hidden_layers,kv_heads,head_dim,head_dim).contiguous()
    return VectorKVCache(compression_ratio=compression_ratio,approx_ratio=0.1,target_budget=target_budget,max_physical_budget=max_physical_budget,min_recent_tokens=16,w_ols=w_ols), VectorAttentionHookManager

def benchmark_variant(name, prompt_text, new_tokens=30, target_budget=256, max_physical_budget=512):
    cache, manager_cls=make_cache(name,target_budget=target_budget,max_physical_budget=max_physical_budget)
    run_model=model
    manager=None
    if manager_cls: manager=manager_cls(model=run_model,kv_cache=cache); manager.register_hooks()
    batch=tokenizer(prompt_text,return_tensors='pt').to(DEVICE)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    with torch.inference_mode():
        start=time.perf_counter(); out=run_model(**batch,past_key_values=cache,use_cache=True); torch.cuda.synchronize()
    prefill_ms=(time.perf_counter()-start)*1000; generated=[]; times=[]
    for _ in range(new_tokens):
        token=out.logits[:,-1:].argmax(-1); generated.append(token.item()); torch.cuda.synchronize(); start=time.perf_counter()
        with torch.inference_mode(): out=run_model(token,past_key_values=cache,use_cache=True)
        torch.cuda.synchronize(); times.append(time.perf_counter()-start)
    if manager: manager.remove_hooks()
    result={'variant':name,'prompt_tokens':batch.input_ids.shape[-1],'new_tokens':new_tokens,'prefill_ms':prefill_ms,'decode_ms':sum(times)/len(times)*1000,'tok_s':new_tokens/sum(times),'persistent_kv_bytes':cache.get_memory_bytes(),'peak_allocated':torch.cuda.max_memory_allocated(),'peak_reserved':torch.cuda.max_memory_reserved(),'physical_len':[cache._physical_len(i) if hasattr(cache,'_physical_len') else cache.get_seq_length(i) for i in range(model.config.num_hidden_layers)],'generated':generated}
    print(result); return result

In [ ]:
# Cell 6 — run comparable variants
CONTEXT_TEXT = PROMPT + ' ' + ('Discuss memory growth, eviction safety, and quality trade-offs. ' * 80)
results=[]
for variant in ['full_fp16','h2o','vector']:
    results.append(benchmark_variant(variant, CONTEXT_TEXT, new_tokens=30, target_budget=256, max_physical_budget=512))
df=pd.DataFrame([{k:v for k,v in r.items() if k!='generated'} for r in results])
df

In [ ]:
# Cell 7 — charts
sns.set_theme(style='whitegrid')
fig, axes=plt.subplots(1,3,figsize=(18,5))
sns.barplot(data=df,x='variant',y='prefill_ms',ax=axes[0]); axes[0].set_title('Prefill latency (ms)')
sns.barplot(data=df,x='variant',y='tok_s',ax=axes[1]); axes[1].set_title('Decode throughput (tok/s)')
sns.barplot(data=df,x='variant',y='persistent_kv_bytes',ax=axes[2]); axes[2].set_title('Persistent KV bytes')
plt.tight_layout(); plt.show()
fig, ax=plt.subplots(figsize=(8,5)); sns.barplot(data=df,x='variant',y='peak_allocated',ax=ax); ax.set_title('Peak allocated CUDA memory'); plt.show()

In [ ]:
# Cell 8 — quality and export
baseline=results[0]['generated']
for item in results[1:]:
    item['first_token_matches_full']=item['generated'][0]==baseline[0]
    item['exact_sequence_match_full']=item['generated']==baseline
    print(item['variant'], {'first_token':item['first_token_matches_full'],'full_sequence':item['exact_sequence_match_full']})
report_dir=ROOT/'reports'; report_dir.mkdir(exist_ok=True)
with open(report_dir/'phase4_real_model_gpu.json','w',encoding='utf-8') as f: json.dump({'model':MODEL_ID,'device':torch.cuda.get_device_name(0),'seed':SEED,'results':results},f,indent=2)
df.to_markdown(report_dir/'phase4_real_model_gpu.md',index=False)
print('reports written to',report_dir)